# TekaRx `gnn-full` on Google Colab

This notebook builds the leakage-controlled FAERS experiment with **2019Q1–2023Q4 for training, 2024Q1 for validation, and 2024Q2 held out for final testing**. It uses Google Drive for durable artifacts and Colab's `/content` disk for DuckDB, memory-mapped graph construction, and training.

> TekaRx outputs are research decision-support signals, not diagnoses or clinical advice. FAERS reports do not establish causality.


## How to run this notebook

Run the work in three restartable stages:

1. **Source staging (CPU runtime):** download and convert immutable sources under Drive.
2. **Cohort and graph build (high-memory CPU runtime):** copy compact inputs to `/content`, build features and the memory-mapped graph, then checkpoint to Drive.
3. **GNN training (GPU runtime):** restore only the graph descriptor and sidecars to `/content`, train with CUDA, and copy the model to Drive.

After a Colab runtime reset, rerun the setup cells before resuming the relevant stage. Do not train directly from the mounted Drive filesystem. Do not add a test-evaluation flag while selecting features or hyperparameters.


## 0. Publish the reviewed code first

The notebook clones GitHub, so the dosage, prospective-split, and memory-mapped graph implementation must already be committed and pushed. Set `GIT_REF` below to the reviewed full commit SHA for a reproducible run. Using `origin/main` is convenient during setup but is not immutable. Never paste a GitHub token into this notebook.


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("This notebook must run in a Google Colab managed runtime.") from exc

drive.mount("/content/drive")


In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

REPO_URL = "https://github.com/matthew-sudo2/Teka-Rx.git"
REPO_DIR = Path("/content/Teka-Rx")
GIT_REF = "origin/main"  # Replace with the reviewed full commit SHA before the final run.

DRIVE_DATA = Path("/content/drive/MyDrive/Teka-Rx-full/data")
LOCAL_DATA = Path("/content/tekarx-data")
MEMORY_LIMIT = "4GB"
THREADS = 2
MATERIALIZATION_BATCH_SIZE = 131_072
XGB_BATCH_SIZE = 65_536
GNN_BATCH_SIZE = 8_192
EDGE_CHUNK_SIZE = 250_000
GNN_SEED = 42
BUILD_DAILYMED = True
MIN_DRIVE_FREE_GIB = 35
RECOMMENDED_DRIVE_FREE_GIB = 80

SPLITS = {
    "train": [f"{year}Q{quarter}" for year in range(2019, 2024) for quarter in range(1, 5)],
    "validation": ["2024Q1"],
    "test": ["2024Q2"],
}
EXPECTED_QUARTERS = tuple(quarter for values in SPLITS.values() for quarter in values)
FAERS_TABLES = ("demo", "drug", "indi", "reac", "outc", "delete")

if len(EXPECTED_QUARTERS) != 22 or len(set(EXPECTED_QUARTERS)) != 22:
    raise RuntimeError(f"The frozen gnn-full plan must contain 22 unique quarters: {EXPECTED_QUARTERS}")
if sys.version_info < (3, 12):
    raise RuntimeError(f"TekaRx requires Python 3.12 or newer; found {sys.version}.")
if DRIVE_DATA == LOCAL_DATA or not str(LOCAL_DATA).startswith("/content/"):
    raise RuntimeError("LOCAL_DATA must be a separate directory on Colab's /content disk.")

DRIVE_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
print(f"Durable data: {DRIVE_DATA}")
print(f"Fast scratch: {LOCAL_DATA}")


In [ ]:
def gibibytes(value: int) -> float:
    return value / (1024**3)


def run_command(*parts: object, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    command = [str(part) for part in parts]
    print("$", shlex.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True, text=True)


def run_tekarx(*parts: object, data_dir: Path) -> subprocess.CompletedProcess[str]:
    return run_command("tekarx", *parts, "--data-dir", data_dir)


def load_json(path: Path) -> dict[str, object]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json_atomic(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def tree_inventory(root: Path) -> dict[str, int]:
    if not root.exists():
        return {}
    return {
        path.relative_to(root).as_posix(): path.stat().st_size
        for path in root.rglob("*")
        if path.is_file()
    }


def tree_size(root: Path) -> int:
    return sum(tree_inventory(root).values())


def copy_tree_verified(source: Path, destination: Path, *, label: str) -> None:
    if not source.is_dir():
        raise FileNotFoundError(f"Missing {label}: {source}")
    source_inventory = tree_inventory(source)
    print(
        f"Copying {label}: {len(source_inventory):,} files, "
        f"{gibibytes(sum(source_inventory.values())):.2f} GiB"
    )
    shutil.copytree(source, destination, dirs_exist_ok=True, copy_function=shutil.copy2)
    mismatches = [
        relative
        for relative, size in source_inventory.items()
        if not (destination / relative).is_file() or (destination / relative).stat().st_size != size
    ]
    if mismatches:
        raise RuntimeError(f"Incomplete {label} copy; first mismatches: {mismatches[:5]}")
    print(f"Verified {label} copy.")


def copy_file_verified(source: Path, destination: Path, *, label: str) -> None:
    if not source.is_file():
        raise FileNotFoundError(f"Missing {label}: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".uploading")
    shutil.copy2(source, temporary)
    if temporary.stat().st_size != source.stat().st_size:
        raise RuntimeError(f"Size mismatch while copying {label}")
    os.replace(temporary, destination)


def assert_faers_scope(root: Path, *, require_complete: bool) -> None:
    expected = set(EXPECTED_QUARTERS)
    problems: list[str] = []
    for table in FAERS_TABLES:
        directory = root / "interim" / "faers" / table
        present = {path.stem for path in directory.glob("*.parquet")} if directory.exists() else set()
        extra = sorted(present - expected)
        missing = sorted(expected - present) if require_complete else []
        if extra:
            problems.append(f"{table}: unexpected quarters {extra}")
        if missing:
            problems.append(f"{table}: missing quarters {missing}")
    raw_root = root / "raw" / "faers"
    if raw_root.exists():
        raw_quarters = {path.name for path in raw_root.iterdir() if path.is_dir() and path.name[:2] == "20"}
        extra_raw = sorted(raw_quarters - expected)
        if extra_raw:
            problems.append(f"raw FAERS: unexpected quarters {extra_raw}")
    if problems:
        raise RuntimeError("FAERS experiment-root check failed:\n  - " + "\n  - ".join(problems))


def require_local_capacity(source_roots: list[Path], *, headroom_gib: float) -> None:
    source_bytes = sum(tree_size(root) for root in source_roots if root.exists())
    free_bytes = shutil.disk_usage("/content").free
    required_bytes = source_bytes + int(headroom_gib * 1024**3)
    print(
        f"Local input copy: {gibibytes(source_bytes):.2f} GiB; "
        f"free: {gibibytes(free_bytes):.2f} GiB; required with headroom: "
        f"{gibibytes(required_bytes):.2f} GiB"
    )
    if free_bytes < required_bytes:
        raise RuntimeError("Insufficient /content space. Choose a larger runtime disk before continuing.")


In [ ]:
local_usage = shutil.disk_usage("/content")
drive_usage = shutil.disk_usage(DRIVE_DATA)
page_size = os.sysconf("SC_PAGE_SIZE")
physical_pages = os.sysconf("SC_PHYS_PAGES")
memory_gib = gibibytes(page_size * physical_pages)

print(f"Python: {platform.python_version()}")
print(f"System RAM: {memory_gib:.2f} GiB")
print(f"/content free: {gibibytes(local_usage.free):.2f} GiB")
drive_free_gib = gibibytes(drive_usage.free)
print(f"Drive free: {drive_free_gib:.2f} GiB")
subprocess.run(["nvidia-smi"], check=False, text=True)

if drive_free_gib < MIN_DRIVE_FREE_GIB:
    raise RuntimeError(
        f"At least {MIN_DRIVE_FREE_GIB} GiB of free Drive space is required; "
        f"found {drive_free_gib:.2f} GiB."
    )
if drive_free_gib < RECOMMENDED_DRIVE_FREE_GIB:
    print(
        f"WARNING: {drive_free_gib:.2f} GiB is below the recommended "
        f"{RECOMMENDED_DRIVE_FREE_GIB} GiB reserve, but above the supported minimum. "
        "Keep only this experiment in the Teka-Rx-full root and monitor space after each stage."
    )
if memory_gib < 12:
    print("WARNING: choose a high-memory runtime before Stage 2 if one is available.")


In [ ]:
if not (REPO_DIR / ".git").is_dir():
    run_command("git", "clone", REPO_URL, REPO_DIR)
run_command("git", "-C", REPO_DIR, "fetch", "--prune", "origin")
run_command("git", "-C", REPO_DIR, "checkout", "--detach", GIT_REF)

required_implementation = (
    REPO_DIR / "src" / "tekarx" / "transform" / "dosage.py",
    REPO_DIR / "src" / "tekarx" / "transform" / "graph_storage.py",
    REPO_DIR / "src" / "tekarx" / "modeling" / "gnn.py",
)
missing_code = [str(path) for path in required_implementation if not path.is_file()]
if missing_code:
    raise RuntimeError(
        "The selected Git ref is the old repository state. Commit and push the reviewed "
        f"implementation first. Missing: {missing_code}"
    )

RESOLVED_GIT_SHA = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
run_command(sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[dev,graph,notebook]")
print(f"Resolved code revision: {RESOLVED_GIT_SHA}")


In [ ]:
import duckdb
import numpy as np
import pyarrow
import torch
import torch_geometric
import xgboost

versions = {
    "python": platform.python_version(),
    "tekarx": importlib.metadata.version("tekarx"),
    "duckdb": duckdb.__version__,
    "numpy": np.__version__,
    "pyarrow": pyarrow.__version__,
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "xgboost": xgboost.__version__,
}
print(json.dumps(versions, indent=2))
print(f"CUDA available in this runtime: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
run_command(sys.executable, "-m", "pip", "check")
run_command(sys.executable, "-m", "ruff", "check", ".", cwd=REPO_DIR)
run_command(sys.executable, "-m", "pytest", "-q", cwd=REPO_DIR)
run_command("tekarx", "build-prospective", "--help")


## 1. Stage immutable sources in Drive

A standard CPU runtime is enough for this stage. The dedicated `Teka-Rx-full` root prevents later FAERS quarters from changing latest-case-version selection. Completed downloads and Parquet tables are checksum-cached, so rerunning these cells reuses verified artifacts. If Colab stops during ZIP extraction, inspect the exact incomplete quarter directory before retrying; the extractor deliberately refuses to overwrite it.


In [ ]:
experiment_plan = {
    "name": "gnn-full",
    "splits": SPLITS,
    "case_group_key": "caseid",
    "latest_version_key": "caseversion",
    "test_policy": "locked until model and hyperparameters are frozen",
}
plan_path = DRIVE_DATA / "experiment_definition.json"
if plan_path.is_file():
    existing_plan = load_json(plan_path)
    if existing_plan.get("name") != experiment_plan["name"] or existing_plan.get("splits") != SPLITS:
        raise RuntimeError(f"Conflicting experiment definition already exists: {plan_path}")
else:
    write_json_atomic(plan_path, experiment_plan)
assert_faers_scope(DRIVE_DATA, require_complete=False)
print(json.dumps(experiment_plan, indent=2))


In [ ]:
run_tekarx("extract-drugcentral", data_dir=DRIVE_DATA)
run_tekarx("build-drugcentral", data_dir=DRIVE_DATA)


In [ ]:
assert_faers_scope(DRIVE_DATA, require_complete=False)
run_tekarx("extract-faers", "--preset", "gnn-full", data_dir=DRIVE_DATA)
run_tekarx("build-faers", "--preset", "gnn-full", data_dir=DRIVE_DATA)
assert_faers_scope(DRIVE_DATA, require_complete=True)


In [ ]:
if BUILD_DAILYMED:
    run_tekarx("extract-dailymed", data_dir=DRIVE_DATA)
    run_tekarx("build-dailymed", data_dir=DRIVE_DATA)
    run_tekarx("build-rxnorm-lookup", data_dir=DRIVE_DATA)
else:
    print("DailyMed/RxNorm staging skipped. DrugCentral exact and synonym mapping remains available.")


In [ ]:
import pandas as pd
import pyarrow.parquet as pq

assert_faers_scope(DRIVE_DATA, require_complete=True)
audit_rows = []
for table in FAERS_TABLES:
    paths = sorted((DRIVE_DATA / "interim" / "faers" / table).glob("*.parquet"))
    audit_rows.append(
        {
            "table": table,
            "quarters": len(paths),
            "rows": sum(pq.ParquetFile(path).metadata.num_rows for path in paths),
            "size_gib": gibibytes(sum(path.stat().st_size for path in paths)),
        }
    )
display(pd.DataFrame(audit_rows))

required_drugcentral = ("structures.parquet", "synonyms.parquet", "struct2atc.parquet")
missing_drugcentral = [
    name
    for name in required_drugcentral
    if not (DRIVE_DATA / "interim" / "drugcentral" / name).is_file()
]
drugcentral_dumps = sorted((DRIVE_DATA / "raw" / "drugcentral").glob("*.sql.gz"))
if missing_drugcentral or len(drugcentral_dumps) != 1:
    raise RuntimeError(
        f"DrugCentral preflight failed: missing={missing_drugcentral}, dumps={drugcentral_dumps}"
    )
source_success = {
    "stage": "sources_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "git_sha": RESOLVED_GIT_SHA,
    "split_preset": "gnn-full",
    "quarters": list(EXPECTED_QUARTERS),
}
write_json_atomic(DRIVE_DATA / "_SOURCES_SUCCESS.json", source_success)
print("Stage 1 complete: all 22 quarters × 6 FAERS tables and DrugCentral inputs verified.")


## 2. Build the cohort, features, and graph on local SSD

Use a high-memory CPU runtime if available, then rerun the setup cells. This stage copies only interim Parquet/reference files and the one DrugCentral SQL dump to `/content`; raw FAERS ZIP/TXT files remain in Drive. Feature artifacts are checkpointed before the graph build so a later runtime can resume from Drive.


In [ ]:
assert_faers_scope(DRIVE_DATA, require_complete=True)
source_success_path = DRIVE_DATA / "_SOURCES_SUCCESS.json"
if not source_success_path.is_file():
    raise RuntimeError("Source checkpoint has no success marker; rerun the Stage 1 audit.")
capacity_sources = [DRIVE_DATA / "interim", DRIVE_DATA / "raw" / "drugcentral"]
feature_success_path = DRIVE_DATA / "processed" / "_FEATURES_SUCCESS.json"
if feature_success_path.is_file():
    feature_success = load_json(feature_success_path)
    if feature_success.get("git_sha") != RESOLVED_GIT_SHA:
        raise RuntimeError("Feature checkpoint was built by a different Git revision. Rebuild it.")
    capacity_sources.append(DRIVE_DATA / "processed")
require_local_capacity(capacity_sources, headroom_gib=30)

copy_tree_verified(
    DRIVE_DATA / "interim", LOCAL_DATA / "interim", label="interim Parquet inputs"
)
copy_tree_verified(
    DRIVE_DATA / "raw" / "drugcentral",
    LOCAL_DATA / "raw" / "drugcentral",
    label="raw DrugCentral dump",
)
if feature_success_path.is_file():
    copy_tree_verified(
        DRIVE_DATA / "processed", LOCAL_DATA / "processed", label="processed checkpoint"
    )
assert_faers_scope(LOCAL_DATA, require_complete=True)


In [ ]:
for stale_marker in ("_FEATURES_SUCCESS.json", "_GRAPH_SUCCESS.json", "_GNN_SUCCESS.json"):
    for checkpoint_root in (DRIVE_DATA / "processed", LOCAL_DATA / "processed"):
        (checkpoint_root / stale_marker).unlink(missing_ok=True)
run_tekarx(
    "build-prospective",
    "--split-preset",
    "gnn-full",
    "--threads",
    THREADS,
    "--memory-limit",
    MEMORY_LIMIT,
    "--skip-graph",
    data_dir=LOCAL_DATA,
)


In [ ]:
cohort_manifest_path = LOCAL_DATA / "processed" / "cohort_manifest.json"
cohort_manifest = load_json(cohort_manifest_path)
if cohort_manifest.get("split_preset") != "gnn-full" or cohort_manifest.get("splits") != SPLITS:
    raise RuntimeError("Cohort manifest does not match the frozen gnn-full split.")
missing_coverage = cohort_manifest.get("quarter_coverage", {}).get("missing", {})
if set(missing_coverage) != set(SPLITS) or any(missing_coverage.values()):
    raise RuntimeError(f"Incomplete cohort quarter coverage: {missing_coverage}")

feature_cohort = LOCAL_DATA / "processed" / "tekarx_cohort_feature_rescue.parquet"
dose_edges = LOCAL_DATA / "processed" / "edges" / "report_drug_dose.parquet"
for required_path in (feature_cohort, dose_edges):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

cohort_sql = feature_cohort.as_posix().replace("'", "''")
connection = duckdb.connect()
try:
    connection.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
    cohort_audit = connection.execute(
        f"""
        SELECT split, count(*) AS rows, count(DISTINCT primaryid) AS primaryids,
               count(DISTINCT caseid) AS caseids, min(quarter) AS first_quarter,
               max(quarter) AS last_quarter
        FROM read_parquet('{cohort_sql}')
        GROUP BY split ORDER BY split
        """
    ).df()
finally:
    connection.close()
display(cohort_audit)
if set(cohort_audit["split"]) != set(SPLITS):
    raise RuntimeError("Unexpected cohort split labels.")
if not (cohort_audit["rows"] == cohort_audit["primaryids"]).all():
    raise RuntimeError("Cohort is not unique by primaryid.")
if not (cohort_audit["rows"] == cohort_audit["caseids"]).all():
    raise RuntimeError("A caseid crossed versions or appears more than once.")

feature_metadata = {
    "stage": "features_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "git_sha": RESOLVED_GIT_SHA,
    "versions": versions,
    "split_preset": "gnn-full",
    "test_evaluated": False,
}
write_json_atomic(LOCAL_DATA / "processed" / "colab_run_metadata.json", feature_metadata)
copy_tree_verified(
    LOCAL_DATA / "processed", DRIVE_DATA / "processed", label="feature checkpoint to Drive"
)
write_json_atomic(
    DRIVE_DATA / "processed" / "_FEATURES_SUCCESS.json",
    {**feature_metadata, "checkpoint": "verified_complete"},
)


In [ ]:
for stale_marker in ("_GRAPH_SUCCESS.json", "_GNN_SUCCESS.json"):
    for checkpoint_root in (DRIVE_DATA / "processed", LOCAL_DATA / "processed"):
        (checkpoint_root / stale_marker).unlink(missing_ok=True)
feature_cohort = LOCAL_DATA / "processed" / "tekarx_cohort_feature_rescue.parquet"
run_tekarx(
    "build-graph",
    "--cohort-path",
    feature_cohort,
    "--graph-storage",
    "memory-mapped",
    "--materialization-batch-size",
    MATERIALIZATION_BATCH_SIZE,
    "--xgb-batch-size",
    XGB_BATCH_SIZE,
    "--xgb-rounds",
    1000,
    "--xgb-early-stopping",
    50,
    "--xgb-max-depth",
    0,
    "--xgb-max-leaves",
    63,
    "--threads",
    THREADS,
    "--memory-limit",
    MEMORY_LIMIT,
    data_dir=LOCAL_DATA,
)


In [ ]:
from tekarx.transform.graph_storage import load_graph_arrays

local_processed = LOCAL_DATA / "processed"
graph_descriptor = local_processed / "tekarx_graph.pt"
graph_manifest_path = local_processed / "graph_manifest.json"
graph_manifest = load_json(graph_manifest_path)
if graph_manifest.get("storage", {}).get("format") != "tekarx.memmap_graph":
    raise RuntimeError("Graph is not using the memory-mapped storage format.")
if "train" not in str(graph_manifest.get("ror_scope", "")).lower():
    raise RuntimeError("Drug ROR was not frozen from training patients.")
if "train" not in str(graph_manifest.get("unknown_vocabulary_scope", "")).lower():
    raise RuntimeError("Unknown-drug vocabulary was not selected from training patients.")
if graph_manifest.get("message_passing", {}).get("held_out_patient_messages_to_shared_drugs") is not False:
    raise RuntimeError("Held-out patient messages can reach shared drug nodes.")
if graph_manifest.get("auxiliary_targets", {}).get("included_in_patient_x") is not False:
    raise RuntimeError("Auxiliary outcome targets leaked into patient features.")

bundle = load_graph_arrays(graph_descriptor, mmap_mode="r")
print(
    {
        name: {"shape": list(array.shape), "dtype": str(array.dtype)}
        for name, array in bundle.arrays.items()
    }
)
print(f"Validation-only XGBoost AUC: {graph_manifest['record']['validation_auc']:.6f}")
del bundle

graph_metadata = {
    **feature_metadata,
    "stage": "graph_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "graph_storage": "tekarx.memmap_graph",
    "xgboost_validation_auc": graph_manifest["record"]["validation_auc"],
}
write_json_atomic(local_processed / "colab_run_metadata.json", graph_metadata)

drive_processed = DRIVE_DATA / "processed"
copy_tree_verified(
    local_processed / "tekarx_graph_arrays",
    drive_processed / "tekarx_graph_arrays",
    label="memory-mapped graph sidecars to Drive",
)
for name in (
    "tekarx_tabular_baseline.npz",
    "xgboost_baseline.json",
):
    copy_file_verified(local_processed / name, drive_processed / name, label=name)
copy_file_verified(graph_descriptor, drive_processed / "tekarx_graph.pt", label="graph descriptor")
copy_file_verified(graph_manifest_path, drive_processed / "graph_manifest.json", label="graph manifest")
copy_file_verified(
    local_processed / "colab_run_metadata.json",
    drive_processed / "colab_run_metadata.json",
    label="Colab run metadata",
)
write_json_atomic(
    drive_processed / "_GRAPH_SUCCESS.json",
    {**graph_metadata, "checkpoint": "verified_complete"},
)
print("Stage 2 complete and checkpointed.")


## 3. Train on a Colab CUDA GPU

Start a GPU runtime, then rerun the mount, configuration, helper, install, and dependency-audit cells. The graph is restored to `/content`; the temporary neighbor matrix is also created there. The current trainer saves atomically after training finishes but does not resume from a partial epoch, so use a runtime expected to remain available for the complete run.


In [ ]:
drive_processed = DRIVE_DATA / "processed"
local_processed = LOCAL_DATA / "processed"
graph_success_path = drive_processed / "_GRAPH_SUCCESS.json"
if not graph_success_path.is_file():
    raise RuntimeError("Drive graph checkpoint is incomplete; its success marker is missing.")
graph_success = load_json(graph_success_path)
if graph_success.get("graph_storage") != "tekarx.memmap_graph":
    raise RuntimeError("Drive graph checkpoint has the wrong storage format.")
if graph_success.get("git_sha") != RESOLVED_GIT_SHA:
    raise RuntimeError("Drive graph checkpoint was built by a different Git revision.")
graph_source_roots = [drive_processed / "tekarx_graph_arrays"]
require_local_capacity(graph_source_roots, headroom_gib=10)
copy_file_verified(
    drive_processed / "tekarx_graph.pt",
    local_processed / "tekarx_graph.pt",
    label="graph descriptor",
)
copy_tree_verified(
    drive_processed / "tekarx_graph_arrays",
    local_processed / "tekarx_graph_arrays",
    label="memory-mapped graph sidecars to local SSD",
)
restored_bundle = load_graph_arrays(local_processed / "tekarx_graph.pt", mmap_mode="r")
print(f"Restored {len(restored_bundle.arrays)} validated graph arrays.")
del restored_bundle


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun setup.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"GPU memory: {gibibytes(gpu.total_memory):.2f} GiB")
print(f"Torch CUDA runtime: {torch.version.cuda}")


In [ ]:
for checkpoint_root in (DRIVE_DATA / "processed", LOCAL_DATA / "processed"):
    (checkpoint_root / "_GNN_SUCCESS.json").unlink(missing_ok=True)
training_arguments = (
    "train-gnn",
    "--device",
    "cuda",
    "--feature-track",
    "prospective",
    "--batch-size",
    GNN_BATCH_SIZE,
    "--edge-chunk-size",
    EDGE_CHUNK_SIZE,
    "--seed",
    GNN_SEED,
    "--epochs",
    100,
    "--patience",
    15,
)
run_tekarx(*training_arguments, data_dir=LOCAL_DATA)


In [ ]:
local_model = local_processed / "tekarx_inductive_gnn.pt"
local_manifest = local_processed / "tekarx_inductive_gnn_manifest.json"
gnn_manifest = load_json(local_manifest)
if gnn_manifest.get("record", {}).get("test_auc") is not None:
    raise RuntimeError("The final test labels were consumed during model selection.")
if gnn_manifest.get("leakage_controls", {}).get("test_evaluated") is not False:
    raise RuntimeError("GNN manifest does not confirm the locked-test protocol.")
if gnn_manifest.get("feature_track") != "prospective":
    raise RuntimeError("Unexpected GNN feature track.")

training_metadata = {
    "stage": "gnn_training_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "git_sha": RESOLVED_GIT_SHA,
    "versions": versions,
    "gpu": gpu.name,
    "torch_cuda": torch.version.cuda,
    "split_preset": "gnn-full",
    "validation_auc": gnn_manifest["record"]["validation_auc"],
    "best_epoch": gnn_manifest["record"]["best_epoch"],
    "test_evaluated": False,
}
write_json_atomic(local_processed / "colab_training_metadata.json", training_metadata)
for name in (
    "tekarx_inductive_gnn.pt",
    "tekarx_inductive_gnn_manifest.json",
    "colab_training_metadata.json",
):
    copy_file_verified(local_processed / name, drive_processed / name, label=name)
write_json_atomic(
    drive_processed / "_GNN_SUCCESS.json",
    {**training_metadata, "checkpoint": "verified_complete"},
)

print(json.dumps(training_metadata, indent=2))
print("Stage 3 complete: validation-selected model safely stored in Drive; test remains locked.")


## Finished

The durable experiment is under `MyDrive/Teka-Rx-full/data/processed/`. Preserve the manifests, graph descriptor, entire `tekarx_graph_arrays/` directory, model, and Colab metadata together. Compare models only with validation metrics until all choices are frozen. Final test evaluation should be a separate, deliberate release step.
